# Route Resilience — 6-Channel clDice Training

U-Net/ResNet34 with 6 input channels `[R, G, B, ExR, Hue, Sat]` + clDice loss.

**Inputs:**
- Model: `training/training/V1` (src files: train.py, model.py, etc.)
- Dataset: `balraj98/deepglobe-road-extraction-dataset`

**Outputs** → `/kaggle/working/`: `best.pt`, `last.pt`, `metrics.csv`, `summary.json`, `curves.png`, prediction grids.

In [ ]:
!pip install -q segmentation-models-pytorch

In [ ]:
import os, sys, glob

# --- Discover where Kaggle mounted the model files ---
# Kaggle Models mount at /kaggle/input/<slug>/<framework>/<variation>/<version>/
hits = glob.glob("/kaggle/input/**/train.py", recursive=True)
assert hits, "train.py not found — check that the 'training' Model is added as input"
SRC_DIR = os.path.dirname(hits[0])
print(f"Source files at: {SRC_DIR}")
print(os.listdir(SRC_DIR))

# --- Discover DeepGlobe dataset ---
DG_CANDIDATES = [
    "/kaggle/input/deepglobe-road-extraction-dataset/train",
    "/kaggle/input/deepglobe-road-extraction-dataset",
]
DATA_DIR = None
for d in DG_CANDIDATES:
    if os.path.isdir(d) and glob.glob(os.path.join(d, "*_sat.jpg")):
        DATA_DIR = d
        break
if DATA_DIR is None:
    # search deeper
    sat = glob.glob("/kaggle/input/**/*_sat.jpg", recursive=True)
    if sat:
        DATA_DIR = os.path.dirname(sat[0])
assert DATA_DIR, "DeepGlobe tiles not found — add the dataset as notebook input"
n_tiles = len(glob.glob(os.path.join(DATA_DIR, "*_sat.jpg")))
print(f"DeepGlobe tiles at: {DATA_DIR}  ({n_tiles} tiles)")

In [ ]:
import yaml, shutil

OUT_DIR = "/kaggle/working/phase1_6ch"
os.makedirs(OUT_DIR, exist_ok=True)

# Load config from model files, patch paths
cfg_src = os.path.join(SRC_DIR, "config.yaml")
cfg = yaml.safe_load(open(cfg_src))

cfg["data"]["train_dir"] = DATA_DIR
cfg["data"]["mass_root"] = None
cfg["train"]["out_dir"] = OUT_DIR
cfg["train"]["device"] = "cuda"

# Write patched config
PATCHED_CFG = "/kaggle/working/config.yaml"
with open(PATCHED_CFG, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Patched config:")
for k in ["data", "train", "model"]:
    print(f"  {k}: {cfg[k]}")

In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Add source dir to path and run training
sys.path.insert(0, SRC_DIR)

# Patch sys.argv for train.py's argparse
sys.argv = [
    "train.py",
    "--config", PATCHED_CFG,
    "--loss", "cldice",
    "--out", OUT_DIR,
]

import train
train.main()

In [ ]:
# --- Show results ---
import json
from IPython.display import display, Image as IPyImage

summary_path = os.path.join(OUT_DIR, "summary.json")
if os.path.exists(summary_path):
    s = json.load(open(summary_path))
    print(f"Best IoU: {s['best_val_iou']:.4f} @ epoch {s['best_epoch']}")
    print(f"Epochs run: {s['epochs_run']}")
    print(f"Final: {s['final']}")

curves = os.path.join(OUT_DIR, "curves.png")
if os.path.exists(curves):
    display(IPyImage(filename=curves))

# Show last prediction grid
preds = sorted(glob.glob(os.path.join(OUT_DIR, "preds_ep*.png")))
if preds:
    display(IPyImage(filename=preds[-1]))

print(f"\nOutputs in {OUT_DIR}:")
for f in os.listdir(OUT_DIR):
    sz = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f"  {f:30s} {sz/1e6:.1f} MB" if sz > 1e6 else f"  {f}")